In [ ]:
%load_ext autoreload
%autoreload 2
import dt4dds_benchmark
import plotly.express as px
import pandas as pd
import numpy as np

data_initcov = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/initcov/{s}.hdf5').get_data() for s in (
    'aeon_low', 'aeon_medium', 'aeon_high', 'rs_low', 'rs_medium', 'rs_high', 'modulation_medium', 'ldpc_medium', 'dbgps_low', 'dbgps_medium', 'dbgps_high',
)])
data_seqdepth = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/seqdepth/{s}.hdf5').get_data() for s in (
    'aeon_low', 'aeon_medium', 'aeon_high', 'rs_low', 'rs_medium', 'rs_high', 'modulation_medium', 'ldpc_medium', 'dbgps_low', 'dbgps_medium', 'dbgps_high',
)])

In [ ]:
colormap_type = {'DNAAeon': '#add1eb', 'DNARS': '#f9bd9f', 'DBGPS': '#31a354', 'LDPC': '#756bb1', 'Modulation': '#636363'}

### define functions to select pareto-optimal points

In [ ]:
# see https://stackoverflow.com/questions/32791911/fast-calculation-of-pareto-front-in-python
def is_pareto_optimal(costs):
    is_efficient = np.all(np.logical_not(np.isnan(costs)), axis=1)
    for i, c in enumerate(costs):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs[is_efficient]<c, axis=1)  # Keep any point with a lower cost
            is_efficient[i] = True  # And keep self
    return is_efficient
    
# function that receives a groupby subset and only returns the pareto front
def apply_pareto(group, cost1, cost2, cost1_max = False, cost2_max = False):
    # get the costs
    costs = group[[cost1, cost2]].values.astype(float)
    # invert the costs if they are maximization problems
    if cost1_max:
        costs[:,0] = -costs[:,0]
    if cost2_max:
        costs[:,1] = -costs[:,1]
    # get the pareto front
    return group[is_pareto_optimal(costs)]

# function that receives a groupby subset and returns the complete pareto front by adding extreme points if not present already
def complete_pareto_front(group):
    full = group.copy()
    if full['workflow.initial_coverage'].max() < 1000:
        add = pd.DataFrame.from_dict({'codec.type': full['codec.type'].iloc[0], 'codec.name': full['codec.name'].iloc[0], 'workflow.type': full['workflow.type'].iloc[0], 'workflow.initial_coverage': 1000, 'workflow.sequencing_depth': 0.999*full['workflow.sequencing_depth'].min()}, orient='index').T
        full = pd.concat([full, add], ignore_index=True).reset_index(drop=True)
    if full['workflow.sequencing_depth'].max() < 1000:
        add = pd.DataFrame.from_dict({'codec.type': full['codec.type'].iloc[0], 'codec.name': full['codec.name'].iloc[0], 'workflow.type': full['workflow.type'].iloc[0], 'workflow.initial_coverage': 0.999*full['workflow.initial_coverage'].min(), 'workflow.sequencing_depth': 1000}, orient='index').T
        full = pd.concat([full, add], ignore_index=True).reset_index(drop=True)
    return full.reset_index(drop=True)

### get the fits for both scenarios, and harmonize the dataframes

In [ ]:
df_initcov = data_initcov.get_fits_by_group(['codec.type', 'codec.name', 'workflow.name', 'workflow.type', 'workflow.initial_coverage'], 'workflow.sequencing_depth', additional_agg={'code_rate': 'mean', 'n_sequences': 'mean', 'sequence_length': 'mean', 'n_bases': 'mean', 'filesize_bit': 'mean'})
df_initcov['workflow.sequencing_depth'] = df_initcov['threshold']

df_seqdepth = data_seqdepth.get_fits_by_group(['codec.type', 'codec.name', 'workflow.name', 'workflow.type', 'workflow.sequencing_depth'], 'workflow.initial_coverage', additional_agg={'code_rate': 'mean', 'n_sequences': 'mean', 'sequence_length': 'mean', 'n_bases': 'mean', 'filesize_bit': 'mean'})
df_seqdepth['workflow.initial_coverage'] = df_seqdepth['threshold']

df = pd.concat([df_initcov, df_seqdepth], ignore_index=True)
df['workflow.sequencing_depth'] = df['workflow.sequencing_depth'].astype(float)
df['workflow.initial_coverage'] = df['workflow.initial_coverage'].astype(float)

df['eff_code_rate'] = df['code_rate'].astype(float) / df['workflow.initial_coverage'].astype(float)
df['eff_storage_density'] = 122.2 * df['code_rate'].astype(float) / df['workflow.initial_coverage'].astype(float)
df['name'] = df['codec.type'] + '-' + df['codec.name']

### apply the pareto filter to remove non-optimal points for initial coverage + sequencing depth

In [ ]:
idf = df.groupby(['codec.type', 'codec.name', 'workflow.type']).apply(apply_pareto, 'workflow.initial_coverage', 'workflow.sequencing_depth', cost1_max = False, cost2_max = False, include_groups=False).reset_index()

In [ ]:
plotdf = idf.groupby(['codec.type', 'codec.name', 'workflow.type'])[['codec.type', 'codec.name', 'workflow.type', 'workflow.initial_coverage', 'workflow.sequencing_depth']].apply(complete_pareto_front).reset_index(drop=True)

plotdf = plotdf.sort_values(['codec.type', 'workflow.sequencing_depth', 'workflow.initial_coverage'], ascending=[False, True, True])
plotdf['codec.name'] = plotdf['codec.name'].replace({'high': 'High', 'medium': 'Medium', 'low': 'Low'})
plotdf['name'] = plotdf['codec.type'] + '-' + plotdf['codec.name']

plotdf

In [ ]:
fig = px.line(
    plotdf,
    x='workflow.initial_coverage', 
    y='workflow.sequencing_depth', 
    log_y=True, 
    log_x=True, 
    color='codec.type', 
    facet_row='workflow.type', 
    facet_row_spacing=0.09,
    facet_col='codec.name',
    facet_col_spacing=0.06,
    markers=True,
    color_discrete_map=colormap_type,
    category_orders={'codec.name': ['High', 'Medium', 'Low'], 'workflow.type': ['BestCase', 'WorstCase']},
    range_x=[0.3, 500],
    range_y=[0.3, 500],
)
fig.update_layout(
    showlegend=False,
    width=340,
    height=240,
    margin=dict(l=0, r=10, t=20, b=0),
)
fig.add_hline(
    y=30,
    line_dash='dash',
    line_width=1,
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(dtick=1)
fig.update_xaxes(title='Initial coverage', row=1)
fig.update_yaxes(title='Sequencing depth', col=1)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.write_image('./figures/pareto_front_cov_by_rate.svg')
fig.show()

# export data
plotdf.to_csv('./figures/pareto_front_cov_by_rate.csv', index=False)